# 05 — Run the complete four-condition generation experiment

The editable `MODEL_KEYS` and `BUNDLE_NAMES` lists define the combinations to run. The outer loop loads each model once, the inner loop processes its selected bundles, and GPU memory is released before the next model. Comma-separated `JAMIA_MODEL_KEYS` and `JAMIA_BUNDLES` variables can override the lists without editing the notebook. The legacy singular variables `JAMIA_MODEL_KEY` and `JAMIA_BUNDLE` remain supported for one-model or one-bundle jobs.

The authentication cell runs before Transformers is imported. It uses a saved Hugging Face credential or `HF_TOKEN` without printing the token. If no credential is available, it opens Hugging Face's secure login flow. For a compute node without outbound access, restart the kernel with `JAMIA_HF_OFFLINE=1`; this skips login and restricts loading to the shared Hugging Face cache. Offline mode works only if notebook 04 or an earlier download has already cached every selected checkpoint.

Records are appended after every query/condition. Before an existing record is skipped, the notebook validates its model key, bundle, run fingerprint, query ID, condition, generated ID, uniqueness, and nonempty report. Thus, an interrupted multi-combination job can be restarted without duplicating completed work. Cross-model or stale records stop only the affected combination when continue-on-error mode is enabled.

Notebook 05 never fine-tunes. It requires notebook 04's audited corrected adapter and independent evaluation for each selected model. CPU offload is permitted for inference and fingerprinted in provenance; disk offload is rejected. If the full loop exceeds the Biowulf wall-time or GPU-memory allocation, divide `MODEL_KEYS` into smaller batches and rerun. The record-level resume safeguard preserves completed work.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
# Hugging Face authentication and network mode. This cell must run before
# importing Transformers or rerun_code.modeling.
hf_offline = os.environ.get("JAMIA_HF_OFFLINE", "0").strip().lower() in {"1", "true", "yes", "on"}
hf_login_enabled = os.environ.get("JAMIA_HF_LOGIN", "1").strip().lower() not in {"0", "false", "no", "off"}
if hf_offline:
    already_imported = [name for name in ("huggingface_hub", "transformers") if name in sys.modules]
    if already_imported:
        raise RuntimeError(
            "Offline mode must be enabled before Hugging Face libraries are imported. "
            f"Already imported={already_imported}. Restart the kernel and rerun from the first cell."
        )
    os.environ["HF_HUB_OFFLINE"] = "1"
    print("Hugging Face offline-cache mode enabled; no login or Hub request will be attempted.")
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
    from huggingface_hub import get_token, login

    environment_token = os.environ.get("HF_TOKEN", "").strip()
    cached_token_available = bool(get_token())
    if environment_token:
        login(
            token=environment_token,
            add_to_git_credential=False,
            skip_if_logged_in=False,
        )
        print("Hugging Face authentication completed using HF_TOKEN.")
    elif cached_token_available:
        print("Using the existing cached Hugging Face credential.")
    elif hf_login_enabled:
        print("No Hugging Face credential was found. Starting the secure login flow.")
        login(add_to_git_credential=False, skip_if_logged_in=True)
        print("Hugging Face authentication completed.")
    else:
        print(
            "Hugging Face login is disabled and no credential was found. "
            "Only public checkpoints can be downloaded."
        )
    del environment_token

In [ ]:
import hashlib, importlib, traceback
from datetime import datetime, timezone
from rerun_code.common import read_jsonl, write_json
from rerun_code.config import sha256_path
import rerun_code.modeling as modeling_module
from rerun_code.modeling import adapter_config, completed_corrected_adapter, load_processor_and_model, release_accelerator_memory, assert_model_identity, resolve_training_base_model
import rerun_code.generation as generation_module
generation_module = importlib.reload(generation_module)
required_generation_api = 5
actual_generation_api = int(getattr(generation_module, "GENERATION_API_VERSION", 0))
if actual_generation_api < required_generation_api:
    raise ImportError(
        "Notebook 05 and rerun_code/generation.py are out of sync. "
        f"Required generation API {required_generation_api}, found {actual_generation_api} at "
        f"{Path(generation_module.__file__).resolve()}. Copy the updated rerun_code/generation.py "
        "to the Biowulf re_run folder, then rerun this cell."
    )
required_resume_repair = 1
actual_resume_repair = int(
    getattr(generation_module, "GENERATION_RESUME_REPAIR_VERSION", 0)
)
if actual_resume_repair < required_resume_repair:
    raise ImportError(
        "Notebook 05 requires the audited failed-record recovery helper. "
        f"Required repair version {required_resume_repair}, found {actual_resume_repair} at "
        f"{Path(generation_module.__file__).resolve()}. Copy the updated "
        "rerun_code/generation.py to the active Biowulf re_run folder, then rerun this cell."
    )
required_phi4_image_compat = 3
actual_phi4_image_compat = int(
    getattr(generation_module, "PHI4_IMAGE_COMPAT_VERSION", 0)
)
if actual_phi4_image_compat < required_phi4_image_compat:
    raise ImportError(
        "Notebook 05 requires the legacy-NumPy Phi-4/SigLIP2 image compatibility fix. "
        f"Required version {required_phi4_image_compat}, found {actual_phi4_image_compat} at "
        f"{Path(generation_module.__file__).resolve()}. Copy the updated "
        "rerun_code/generation.py to the active Biowulf re_run folder, restart the kernel, "
        "and rerun this cell."
    )
required_phi4_cache_compat = 1
actual_phi4_cache_compat = int(
    getattr(generation_module, "PHI4_CACHE_COMPAT_VERSION", 0)
)
if actual_phi4_cache_compat < required_phi4_cache_compat:
    raise ImportError(
        "Notebook 05 requires the Phi-4 DynamicCache compatibility fix. "
        f"Required version {required_phi4_cache_compat}, found {actual_phi4_cache_compat} at "
        f"{Path(generation_module.__file__).resolve()}. Copy the updated "
        "rerun_code/generation.py to the active Biowulf re_run folder, restart the kernel, "
        "and rerun this cell."
    )
required_phi4_verifier_compat = 1
actual_phi4_verifier_compat = int(
    getattr(generation_module, "PHI4_VERIFIER_COMPAT_VERSION", 0)
)
if actual_phi4_verifier_compat < required_phi4_verifier_compat:
    raise ImportError(
        "Notebook 05 requires the Phi-4 BatchEncoding verifier compatibility fix. "
        f"Required version {required_phi4_verifier_compat}, found "
        f"{actual_phi4_verifier_compat} at "
        f"{Path(generation_module.__file__).resolve()}. Copy the updated "
        "rerun_code/generation.py to the active Biowulf re_run folder, restart the kernel, "
        "and rerun this cell."
    )
required_phi4_empty_recovery = 1
actual_phi4_empty_recovery = int(
    getattr(generation_module, "PHI4_EMPTY_OUTPUT_RECOVERY_VERSION", 0)
)
if actual_phi4_empty_recovery < required_phi4_empty_recovery:
    raise ImportError(
        "Notebook 05 requires the audited Phi-4 empty-output recovery. "
        f"Required version {required_phi4_empty_recovery}, found "
        f"{actual_phi4_empty_recovery} at "
        f"{Path(generation_module.__file__).resolve()}. Copy the updated "
        "rerun_code/generation.py to the active Biowulf re_run folder, restart the kernel, "
        "and rerun this cell."
    )
ModelRunner = generation_module.ModelRunner
CONDITIONS = generation_module.CONDITIONS
run_condition = generation_module.run_condition
validated_completed_generation_ids = generation_module.validated_completed_generation_ids
from rerun_code.leakage_safe_retrieval import load_query_bundle

# Edit either list to run only a subset. The order is preserved.
MODEL_KEYS = [
    "qwen2_1_5b",
    "medgemma_4b",
    "phi4_multimodal",
    "gpt_oss_20b",
    "llama_3_1_8b",
    "medqwen2_7b",
    "openbiollm_8b",
]
BUNDLE_NAMES = ["mimic", "iuhn", "combined"]

def selected_values(plural_env, singular_env, defaults):
    if os.environ.get(plural_env):
        values = [value.strip() for value in os.environ[plural_env].split(",") if value.strip()]
    elif os.environ.get(singular_env):
        values = [os.environ[singular_env].strip()]
    else:
        values = list(defaults)
    if not values:
        raise ValueError(f"No values selected for {plural_env}")
    if len(values) != len(set(values)):
        raise ValueError(f"Duplicate values selected for {plural_env}: {values}")
    return values

model_keys = selected_values("JAMIA_MODEL_KEYS", "JAMIA_MODEL_KEY", MODEL_KEYS)
bundle_names = selected_values("JAMIA_BUNDLES", "JAMIA_BUNDLE", BUNDLE_NAMES)
n_items = int(os.environ["JAMIA_N_ITEMS"]) if os.environ.get("JAMIA_N_ITEMS") else None
progress_every = max(1, int(os.environ.get("JAMIA_PROGRESS_EVERY", "10")))
continue_on_error = os.environ.get("JAMIA_CONTINUE_ON_ERROR", "1").strip().lower() not in {"0", "false", "no"}
unknown_models = sorted(set(model_keys) - set(CONFIG["models"]))
unknown_bundles = sorted(set(bundle_names) - {"mimic", "iuhn", "combined"})
if unknown_models: raise KeyError(f"Unknown model keys: {unknown_models}")
if unknown_bundles: raise KeyError(f"Unknown bundle names: {unknown_bundles}")
if n_items is not None and n_items <= 0: raise ValueError("JAMIA_N_ITEMS must be positive")
print("Selected models:", model_keys)
print("Selected bundles:", bundle_names)
print("Smoke-test query limit:", n_items)
print("Continue after an isolated combination error:", continue_on_error)

def load_model_context(model_key):
    spec = CONFIG["models"][model_key]
    canonical_base = resolve_training_base_model(spec)
    model_out = PATHS["verifiers"] / model_key
    adapter_audit = completed_corrected_adapter(model_out, model_key, canonical_base, spec=spec)
    if adapter_audit is None:
        raise FileNotFoundError(
            f"No completed corrected adapter found in {model_out}. Run notebook 04 for {model_key}."
        )
    corrected_adapter = adapter_audit["adapter_dir"]
    adapter_weights_file = adapter_audit["weights_file"]
    corrected_adapter_config = adapter_config(corrected_adapter)
    training_provenance_path = model_out / "training_provenance.json"
    independent_metrics_path = model_out / "independent_test_metrics.json"
    independent_predictions_path = model_out / "independent_test_predictions.jsonl"
    missing = [str(path) for path in (independent_metrics_path, independent_predictions_path) if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Notebook 04 independent evaluation is incomplete for "
            f"{model_key}. Missing={missing}"
        )
    independent_metrics = json.loads(independent_metrics_path.read_text(encoding="utf-8"))
    independent_predictions = read_jsonl(independent_predictions_path)
    if int(independent_metrics.get("n") or 0) <= 0:
        raise AssertionError(f"Invalid independent verifier sample count for {model_key}")
    if int(independent_metrics["n"]) != len(independent_predictions):
        raise AssertionError(
            f"Independent verifier metrics for {model_key} report n={independent_metrics['n']}, "
            f"but predictions contain {len(independent_predictions)} records"
        )

    print("\nLoading model:", model_key, spec["display_name"])
    print("Canonical backbone:", canonical_base)
    print("Corrected adapter:", corrected_adapter)
    release_accelerator_memory()
    processor, tokenizer, model, actual_base = load_processor_and_model(
        spec, adapter_path=corrected_adapter, base_model_override=canonical_base,
    )
    assert_model_identity(model_key, spec, actual_base)
    if not getattr(model, "peft_config", None):
        raise AssertionError(f"Corrected LoRA was not attached for {model_key}")
    loaded_backbone = model.get_base_model() if hasattr(model, "get_base_model") else model
    backbone_load_method = getattr(loaded_backbone, "_jamia_load_method", "transformers_from_pretrained")
    backbone_device_map = {
        str(key): str(value)
        for key, value in (getattr(loaded_backbone, "hf_device_map", {}) or {}).items()
    }
    evaluation_cpu_offload = any(value.lower() == "cpu" for value in backbone_device_map.values())
    runner = ModelRunner(
        processor, tokenizer, model, spec["architecture"], CONFIG["generation"],
        actual_base=actual_base,
    )
    if model_key == "phi4_multimodal":
        print(
            "Phi-4 DynamicCache compatibility targets:",
            list(runner.phi4_dynamic_cache_targets),
        )
    return {
        "model_key": model_key, "spec": spec, "canonical_base": canonical_base,
        "actual_base": actual_base, "model_out": model_out,
        "corrected_adapter": corrected_adapter,
        "adapter_weights_file": adapter_weights_file,
        "corrected_adapter_config": corrected_adapter_config,
        "training_provenance_path": training_provenance_path,
        "independent_metrics_path": independent_metrics_path,
        "independent_predictions_path": independent_predictions_path,
        "independent_metrics": independent_metrics,
        "backbone_load_method": backbone_load_method,
        "backbone_device_map": backbone_device_map,
        "evaluation_cpu_offload": evaluation_cpu_offload,
        "runner": runner,
    }

def run_model_bundle(context, bundle_name):
    model_key, spec = context["model_key"], context["spec"]
    _, all_queries = load_query_bundle(PATHS["bundles"] / bundle_name / "queries")
    queries = all_queries[:n_items] if n_items is not None else all_queries
    neighbors_path = PATHS["bundles"] / bundle_name / "neighbors.jsonl"
    neighbors_by_query = {row["query_record_id"]: row["neighbors"] for row in read_jsonl(neighbors_path)}
    missing_neighbors = sorted({row["record_id"] for row in queries} - set(neighbors_by_query))
    if missing_neighbors:
        raise KeyError(f"Missing neighbors for {model_key}/{bundle_name}: {missing_neighbors[:10]}")
    output_dir = PATHS["generation"] / model_key / bundle_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "results.jsonl"

    run_provenance = {
        "model_key": model_key,
        "bundle": bundle_name,
        "canonical_base_model": context["canonical_base"],
        "actual_base_model": context["actual_base"],
        "architecture": spec["architecture"],
        "model_display_name": spec["display_name"],
        "loader_profile": spec.get("loader_profile", "default"),
        "tokenizer_source": spec.get("tokenizer_source", "base"),
        "chat_template_policy": spec.get("chat_template_policy", "auto"),
        "reference_loading_notebook": spec.get("reference_loading_notebook"),
        "manuscript_identity_note": spec.get("manuscript_identity_note"),
        "corrected_adapter": str(context["corrected_adapter"]),
        "adapter_recorded_base_model": context["corrected_adapter_config"].get("base_model_name_or_path"),
        "adapter_config_sha256": sha256_path(context["corrected_adapter"] / "adapter_config.json"),
        "adapter_weights_file": context["adapter_weights_file"].name,
        "adapter_weights_sha256": sha256_path(context["adapter_weights_file"]),
        "training_provenance_sha256": sha256_path(context["training_provenance_path"]),
        "independent_test_metrics": context["independent_metrics"],
        "independent_test_metrics_sha256": sha256_path(context["independent_metrics_path"]),
        "independent_test_predictions_sha256": sha256_path(context["independent_predictions_path"]),
        "modeling_code_sha256": sha256_path(Path(modeling_module.__file__).resolve()),
        "backbone_load_method": context["backbone_load_method"],
        "backbone_device_map": context["backbone_device_map"],
        "evaluation_cpu_offload": context["evaluation_cpu_offload"],
        "generation_config": CONFIG["generation"],
        "max_revision_passes": CONFIG["max_revision_passes"],
        "neighbors_sha256": sha256_path(neighbors_path),
    }
    if model_key == "phi4_multimodal":
        run_provenance.update({
            "generation_api_version": actual_generation_api,
            "generation_code_sha256": sha256_path(Path(generation_module.__file__).resolve()),
            "phi4_generation_profile": "reference_chat_image_cache_batchencoding_empty_recovery_v6",
            "phi4_image_compat_version": actual_phi4_image_compat,
            "phi4_cache_compat_version": actual_phi4_cache_compat,
            "phi4_verifier_compat_version": actual_phi4_verifier_compat,
            "phi4_empty_output_recovery_version": actual_phi4_empty_recovery,
            "phi4_empty_retry_min_new_tokens": (
                generation_module.PHI4_EMPTY_RETRY_MIN_NEW_TOKENS
            ),
            "phi4_empty_retry_max_prompt_chars": (
                generation_module.PHI4_EMPTY_RETRY_MAX_PROMPT_CHARS
            ),
            "phi4_dynamic_cache_targets": list(
                context["runner"].phi4_dynamic_cache_targets
            ),
        })
    run_id = hashlib.sha256(json.dumps(run_provenance, sort_keys=True).encode("utf-8")).hexdigest()
    run_provenance["run_id"] = run_id
    provenance_file = output_dir / "run_provenance.json"
    if provenance_file.exists():
        previous = json.loads(provenance_file.read_text(encoding="utf-8"))
        if previous != run_provenance:
            has_results = output_file.exists() and any(
                line.strip() for line in output_file.read_text(encoding="utf-8").splitlines()
            )
            if has_results:
                raise RuntimeError(
                    f"Existing output provenance differs for {model_key}/{bundle_name}: {provenance_file}. "
                    "Move that bundle output aside or restore the prior code/adapter."
                )
            write_json(provenance_file, run_provenance)
            print(
                f"Refreshed provenance for zero-record output {model_key}/{bundle_name}; "
                "no generated records were mixed."
            )
    elif output_file.exists() and output_file.stat().st_size:
        raise RuntimeError(f"Existing results lack a run fingerprint: {output_file}")
    else:
        write_json(provenance_file, run_provenance)

    completed, resume_audit = validated_completed_generation_ids(
        output_file, model_key=model_key, bundle_name=bundle_name, run_id=run_id,
        valid_query_ids=[row["record_id"] for row in all_queries], conditions=CONDITIONS,
        repair_failed_records=True,
    )
    requested_ids = {
        f"{model_key}|{bundle_name}|{query['record_id']}|{condition}"
        for query in queries for condition in CONDITIONS
    }
    pending_ids = requested_ids - completed
    resume_audit.update({
        "n_all_bundle_queries": len(all_queries), "n_requested_queries": len(queries),
        "n_requested_records": len(requested_ids),
        "n_already_complete_requested": len(completed & requested_ids),
        "n_pending_requested": len(pending_ids),
    })
    write_json(output_dir / "resume_audit.json", resume_audit)
    print(f"\n{model_key}/{bundle_name}")
    print(json.dumps(resume_audit, indent=2))

    skipped_existing = 0
    generated_new = 0
    for query in queries:
        query_id = query["record_id"]
        neighbors = neighbors_by_query[query_id]
        for condition in CONDITIONS:
            generation_id = f"{model_key}|{bundle_name}|{query_id}|{condition}"
            if generation_id in completed:
                skipped_existing += 1
                continue
            result = run_condition(
                context["runner"], condition, query, neighbors,
                max_revision_passes=CONFIG["max_revision_passes"],
            )
            if result.get("empty_output") is True or not str(result.get("final_report") or "").strip():
                raise RuntimeError(
                    f"Generation returned no usable report for {generation_id}; "
                    "the failed record was not saved. "
                    f"Recovery audit={result.get('empty_output_recovery_by_pass')}"
                )
            record = {
                "generation_record_id": generation_id, "query_record_id": query_id,
                "model_key": model_key, "model_display_name": spec["display_name"],
                "actual_base_model": context["actual_base"], "run_id": run_id,
                "backbone_load_method": context["backbone_load_method"],
                "evaluation_cpu_offload": context["evaluation_cpu_offload"],
                "corrected_adapter_sha256": run_provenance["adapter_weights_sha256"],
                "bundle": bundle_name, "source_dataset": query.get("dataset"),
                "condition": condition, "patient_key": query.get("patient_key") or query_id,
                "patient_id_reliable": bool(query.get("patient_id_reliable", False)),
                "reference_report": query.get("report_text", ""),
                "reference_labels_json": query.get("labels_raw", []),
                "reference_labels_13_manifest_check": query.get("labels_13", []),
                "ground_truth_source": "paired_json.labels",
                "image_path": query.get("image_path", ""),
                "neighbor_record_ids": [row["record_id"] for row in neighbors],
                "neighbor_scores": [row["cosine_score"] for row in neighbors],
                **result,
            }
            with output_file.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                handle.flush()
            completed.add(generation_id)
            generated_new += 1
            if generated_new % progress_every == 0 or generated_new == len(pending_ids):
                print(
                    f"Progress {model_key}/{bundle_name}: generated={generated_new}/"
                    f"{len(pending_ids)}, skipped={skipped_existing}"
                )
    remaining = requested_ids - completed
    if remaining:
        raise AssertionError(
            f"{model_key}/{bundle_name} ended with {len(remaining)} missing records. "
            f"Examples={sorted(remaining)[:10]}"
        )
    summary = {
        **resume_audit, "status": "complete",
        "n_skipped_existing_this_run": skipped_existing,
        "n_generated_this_run": generated_new,
        "n_complete_requested_after_run": len(requested_ids),
        "n_pending_requested_after_run": 0,
    }
    write_json(output_dir / "resume_audit.json", summary)
    print(
        f"Complete {model_key}/{bundle_name}: requested={len(requested_ids)}, "
        f"skipped={skipped_existing}, generated={generated_new}"
    )
    return summary

In [ ]:
orchestration_started = datetime.now(timezone.utc).isoformat()
combination_summaries = []
failures = []
for model_key in model_keys:
    context = None
    try:
        context = load_model_context(model_key)
    except Exception as exc:
        failure = {
            "model_key": model_key, "bundle": None, "stage": "model_load",
            "error_type": type(exc).__name__, "error": str(exc),
            "traceback": traceback.format_exc(),
        }
        failures.append(failure)
        print("MODEL LOAD FAILED:", json.dumps(failure, indent=2))
        release_accelerator_memory()
        if not continue_on_error: raise
        continue
    try:
        for bundle_name in bundle_names:
            try:
                combination_summaries.append(run_model_bundle(context, bundle_name))
            except Exception as exc:
                failure = {
                    "model_key": model_key, "bundle": bundle_name, "stage": "generation",
                    "error_type": type(exc).__name__, "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
                failures.append(failure)
                print("COMBINATION FAILED:", json.dumps(failure, indent=2))
                if not continue_on_error: raise
    finally:
        runner_to_release = context.pop("runner", None)
        if runner_to_release is not None:
            runner_to_release.model = None
            runner_to_release.processor = None
            runner_to_release.tokenizer = None
        del runner_to_release
        context.clear()
        del context
        release_accelerator_memory()
        print("Released model memory for:", model_key)

orchestration = {
    "started_at_utc": orchestration_started,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_models": model_keys, "selected_bundles": bundle_names,
    "n_items_per_bundle": n_items, "continue_on_error": continue_on_error,
    "n_combinations_requested": len(model_keys) * len(bundle_names),
    "n_combinations_complete": len(combination_summaries),
    "n_failures": len(failures), "combinations": combination_summaries,
    "failures": failures,
}
orchestration_path = PATHS["generation"] / "notebook05_orchestration_summary.json"
write_json(orchestration_path, orchestration)
print("Orchestration summary:", orchestration_path)
print(json.dumps({key: value for key, value in orchestration.items() if key not in {"combinations", "failures"}}, indent=2))
if failures:
    failed_names = [f"{row['model_key']}/{row['bundle'] or '*'}" for row in failures]
    failure_details = "\n\n".join(
        f"[{row['model_key']}/{row['bundle'] or '*'}] "
        f"{row['stage']} — {row['error_type']}: {row['error']}\n"
        f"{row['traceback']}"
        for row in failures
    )
    raise RuntimeError(
        f"Generation loop completed with {len(failures)} failed model/bundle entries: {failed_names}. "
        f"Successful records remain saved and will be skipped on restart. See {orchestration_path}.\n\n"
        f"Underlying failures:\n{failure_details}"
    )